In [216]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [217]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [218]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,-915853466 | INFO | 2516037 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc
,-915853452 | INFO | 2516037 - Setting seeds to: 0
!,-915853438 | INFO | 2516037 - REMOVAL TYPE SET AS edge
Pe�6,-915853438 | INFO | 2516037 - Caching System: False.
8]�6,-915853422 | INFO | 2516037 - Instantiating: torch_geometric.datasets.Planetoid
0i��,-915853363 | INFO | 2516037 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
���,-915853360 | INFO | 2516037 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
8]�6,-915853320 | INFO | 2516037 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
0��S,-915853267 | INFO | 2516037 - ['all', 'all_shuffled', '-']
8]�6,-915853266 | INFO | 2516037 - Instan

In [219]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

8]�6,-915853154 | INFO | 2516037 - Instantiating: erasure.model.graphs.SGC.SGC
8]�6,-915853152 | INFO | 2516037 - Instantiating: torch.optim.Adam
8]�6,-915853151 | INFO | 2516037 - Instantiating: torch.nn.CrossEntropyLoss
graph has edges  torch.Size([2, 9104])
��t,-915853026 | INFO | 2516037 - epoch = 0 ---> loss = 1.8022	 accuracy = 0.1361
graph has edges  torch.Size([2, 9104])
��t,-915852912 | INFO | 2516037 - epoch = 1 ---> loss = 2.1026	 accuracy = 0.7428
graph has edges  torch.Size([2, 9104])
��t,-915852794 | INFO | 2516037 - epoch = 2 ---> loss = 1.6855	 accuracy = 0.7912
graph has edges  torch.Size([2, 9104])
��t,-915852680 | INFO | 2516037 - epoch = 3 ---> loss = 1.0872	 accuracy = 0.8213
graph has edges  torch.Size([2, 9104])
��t,-915852564 | INFO | 2516037 - epoch = 4 ---> loss = 0.9783	 accuracy = 0.8376
graph has edges  torch.Size([2, 9104])
��t,-915852449 | INFO | 2516037 - epoch = 5 ---> loss = 0.9715	 accuracy = 0.8392
graph has edges  torch.Size([2, 9104])
��t,-91

In [220]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [221]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [222]:
import torch
print(torch.version.cuda)

12.6


In [223]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.5
    lr: 0.24999999999999994
    maximize: False
    weight_decay: 0
)


In [224]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [225]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

0i��,-915850354 | INFO | 2516037 - Created Configurable: erasure.unlearners.certified_graph_unlearners.CEU.CEU
graph tensor([[2899, 2899, 2899,  ..., 3324, 3325, 3326],
        [ 927, 1068, 1094,  ..., 2820, 1643,   33]])
0i��,-915850260 | INFO | 2516037 - Created Configurable: erasure.unlearners.GoldModel.GoldModelGraph
0i��,-915850230 | INFO | 2516037 - Created Configurable: erasure.unlearners.composite.Identity


In [226]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [227]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [228]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [229]:
print(len( data_manager.partitions['forget']) )

8193


In [230]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

0i��,-915849821 | INFO | 2516037 - Created Configurable: erasure.evaluations.running.RunTime
�,-915849819 | INFO | 2516037 - Function: <function accuracy_score at 0x7f0ca5ded310>
0i��,-915849818 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
,-915849817 | INFO | 2516037 - Function: <function accuracy_score at 0x7f0ca5ded310>
0i��,-915849817 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
��,-915849816 | INFO | 2516037 - Function: <function accuracy_score at 0x7f0ca5ded310>
0i��,-915849815 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
���,-915849814 | INFO | 2516037 - Function: <function accuracy_score at 0x7f0ca5ded310>
0i��,-915849813 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
0i��,-915849812 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.SaveValues
0i��,-915849812 | INFO | 2

/NFSHOME/adangelo/LinkInference/erasure/unlearners/certified_graph_unlearners/CEU.py:78: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)
/NFSHOME/adangelo/LinkInference/erasure/unlearners/certified_graph_unlearners/CEU.py:189: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  all_edges = torch.tensor( data[0][0].edge_index.t().tolist() )
/NFSHOME/adangelo/LinkInference/erasure/unlearners/certified_graph_unlearners/CEU.py:322: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  def inverse_hvp_cg_sgc(self,data, 

�	��,-915838073 | INFO | 2516037 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: True: 0.6846846846846847 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0ca58c6f40>
�	��,-915837990 | INFO | 2516037 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: True: 0.6861861861861862 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0ca4801d30>
�	��,-915837916 | INFO | 2516037 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: False: 0.6846846846846847 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0ca58c6f40>
�	��,-915837759 | INFO | 2516037 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: False: 0.7267267267267268 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0ca4801d30>
8B��,-915837711 | INFO | 2516037 - ####		 Evaluating Unlearner Gol